In [1]:
import asyncio
import json

import pandas as pd

from app.utils.path_util import get_project_root

# -----------------------------------------
# 0. Load labeled_ghost_domains.jsonl
# -----------------------------------------

input_path = get_project_root() / "notebooks/labeled_ghost_domains_training_data.jsonl"

raw_items = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        # each line is {"domain": bool}
        domain, label_bool = next(iter(obj.items()))
        raw_items.append({"domain": domain, "label": int(label_bool)})

len(raw_items), raw_items[:3]



(117,
 [{'domain': 'rfacapitalcorp.com', 'label': 0},
  {'domain': 'mygoodhorse.weebly.com', 'label': 0},
  {'domain': 'abcbni.com', 'label': 1}])

In [2]:
from notebooks.ml_fetch_orchestrator import fetch_all_html_orchestrator

# -----------------------------------------
# 1. Fetch HTML for all domains
# -----------------------------------------

domains = [item["domain"] for item in raw_items]

import nest_asyncio

nest_asyncio.apply()
html_list = asyncio.run(fetch_all_html_orchestrator(domains))

# -----------------------------------------
# Build dataframe
# -----------------------------------------
from df_classifier import DFClassifier

label_map = {item["domain"]: item["label"] for item in raw_items}

features_list = [
    DFClassifier.build_feature_row(domain, html, label_map.get(domain))
    for domain, html in zip(domains, html_list)
]

df = pd.DataFrame(features_list)

df.head()

output_path = "labeled_ghost_domains.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({
            "domain": row["domain"],
            "label": int(row["label"]),
            "text_len": int(row["text_len"]),
            "num_words": int(row["num_words"]),
            "num_internal_links": int(row["num_internal_links"]),
            "has_phone": int(row["has_phone"]),
            "has_email": int(row["has_email"]),
            "has_social": int(row["has_social"]),
            "title_matches_domain": int(row["title_matches_domain"]),
            "has_parked_keywords": int(row["has_parked_keywords"]),
            "has_scam_keywords": int(row["has_scam_keywords"]),
            "is_empty_html": int(row["is_empty_html"]),
        }) + "\n")

df.head()


[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://besttechnology.com (len=1396), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://softwareonline.com (len=1396), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://example.org (len=528), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://bestwish.com (len=114), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://leistung.com (len=114), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://china888.com (len=114), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://atenas.com (len=114), retrying once...
[2026-06-01 12:30:47] [WARNING] [MainThread] Suspicious tiny-body 200 for https://example.net (len=528), retrying once...
[2026-

,text_len,num_words,num_internal_links,has_phone,has_email,has_social,title_matches_domain,has_parked_keywords,has_scam_keywords,is_empty_html,domain,label
0,46,8,0,0,0,0,0,0,0,1,rfacapitalcorp.com,0
1,29,4,0,0,0,0,0,0,0,1,mygoodhorse.weebly.com,0
2,111,15,0,0,0,0,1,0,0,0,abcbni.com,1
3,70,11,1,0,0,0,0,0,0,0,bhmedicalsupplies.com,0
4,139,21,1,0,0,0,0,0,0,0,whymessa.com,0


In [3]:
# ============================================
# 2. Define feature columns
# ============================================

feature_cols = [
    "text_len",
    "num_words",
    "num_internal_links",
    "has_phone",
    "has_email",
    "has_social",
    "title_matches_domain",
    "has_parked_keywords",
    "has_scam_keywords",
    "is_empty_html",
]

X = df[feature_cols].astype(float).values
y = df["label"].astype(int).values

In [19]:
# ============================================
# 3. Train logistic regression
# ============================================

# noinspection PyPackageRequirements
from sklearn.linear_model import LogisticRegression
# noinspection PyPackageRequirements
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",        # best for dense numeric features
    l1_ratio=0,
)

clf.fit(X_scaled, y)

print("Training accuracy:", clf.score(X_scaled, y))

Training accuracy: 0.8376068376068376


In [5]:
# ============================================
# 4. Export pure-Python weights
# ============================================

weights = dict(zip(feature_cols, clf.coef_[0]))
bias = float(clf.intercept_[0])

export = {
    "bias": bias,
    "weights": weights,
    "feature_order": feature_cols,
}

output_path = get_project_root() / "notebooks/ghost_classifier_weights.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2)

print("Exported ghost_classifier_weights.json")


Exported ghost_classifier_weights.json


In [6]:
# ============================================
# Classify suspicious sites list
# ============================================

suspicious_path = get_project_root() / "notebooks/suspicious_sites.json"
# ============================================
# 1. Load suspicious sites list
# ============================================

with open(suspicious_path, "r", encoding="utf-8") as f:
    suspicious_data = json.load(f)

new_domains = suspicious_data["data"]
len(new_domains), new_domains[:5]


# ============================================
# 2. Fetch HTML for all suspicious domains
# ============================================

nest_asyncio.apply()
html_list = asyncio.run(fetch_all_html_orchestrator(new_domains))


# ============================================
# 3. Convert HTML → semantic feature rows
# ============================================

from notebooks.df_classifier import DFClassifier

feature_rows = [
    DFClassifier.build_feature_row(domain, html, label=0)
    for domain, html in zip(new_domains, html_list)
]

df = pd.DataFrame(feature_rows)
df.head()


# ============================================
# 4. Classify using feature-based ghost classifier
# ============================================

from ghost_classifier import is_ghost_features   # NEW: feature-based inference

feature_cols = [
    "text_len", "num_words", "num_internal_links",
    "has_phone", "has_email", "has_social",
    "title_matches_domain", "has_parked_keywords",
    "has_scam_keywords", "is_empty_html",
]

df["ghost_score"] = df[feature_cols].apply(
    lambda row_line: is_ghost_features(row_line.values, threshold=0.0),
    axis=1
)

df["ghost"] = df["ghost_score"] > 0.5
df.head()


# ============================================
# 5. Save classification results
# ============================================

new_output_path = get_project_root() / "notebooks/suspicious_sites_classified.jsonl"

with open(new_output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({
            "domain": row["domain"],
            "ghost": bool(row["ghost"]),
            "score": float(row["ghost_score"])
        }) + "\n")

new_output_path

WindowsPath('D:/DEV/Python/company-data-api/notebooks/suspicious_sites_classified.jsonl')